# ModelForge CUDA PEFT receipt (Colab)

Produces hire-facing **`peft_gpu.json`** with `cuda=true` and a measured schema-pass delta.

**Runtime → Change runtime type → GPU** (T4+).

**Honesty**
- Refuses to run without CUDA.
- This is a real QLoRA micro-run on TinyLlama (not `peft_smoke`).
- Full DomainForge 7B S0/S3/S4 ladder still preferred on RunPod for panel depth.
- **`vllm_cuda.json` is not produced here** — use RunPod `scripts/one_shot_gpu_receipts.sh` for upstream vLLM (Architecture Lab Path B is not a substitute).

In [ ]:
import json, re, subprocess, time
from datetime import datetime, timezone
from pathlib import Path

import torch
assert torch.cuda.is_available(), "Enable a GPU runtime — refuse CPU receipts"
GPU_NAME = torch.cuda.get_device_name(0)
print("CUDA OK:", GPU_NAME)
SMI = subprocess.check_output(["nvidia-smi"], text=True)
print(SMI[:800])
OUT = Path("/content/modelforge_receipts")
OUT.mkdir(exist_ok=True)
(OUT / "_nvidia_smi.txt").write_text(SMI)

In [ ]:
%pip install -q "transformers>=4.40" "peft>=0.11" "trl>=0.9" "bitsandbytes>=0.43" datasets accelerate

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load_model():
    m = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
    return m

base_model = load_model()

PROMPTS = [
    'Classify as JSON with keys intent, priority.\nMessage: "Double charged on March invoice."',
    'Return JSON {"answer": "...", "cite": null} if you lack a citation.\nQ: unused vacation payout for DE contractors?',
    'Reply in under 40 words acknowledging a VIP outage since 09:00 UTC.',
]

def generate(model, prompt, max_new=64):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def schema_pass(text, keys):
    if not keys:
        return len(text.split()) <= 80
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < 0:
        return False
    try:
        obj = json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return False
    return all(k in obj for k in keys)

EXPECT = [["intent", "priority"], ["answer", "cite"], []]

def eval_schema(model):
    ok = 0
    for prompt, keys in zip(PROMPTS, EXPECT):
        text = generate(model, prompt)
        ok += int(schema_pass(text, keys))
        print("---", ok, text[:120].replace("\n", " "))
    return ok / len(PROMPTS)

print("S0 (base) eval...")
s0 = eval_schema(base_model)
print("S0_schema_pass", s0)

In [ ]:
model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(
    model,
    LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"),
)
rows = [
    {
        "text": (
            "### Instruction:\nClassify as JSON with keys intent, priority.\n"
            'Message: "Double charged on March invoice."\n'
            '### Response:\n{"intent":"billing","priority":"high"}'
        )
    },
    {
        "text": (
            "### Instruction:\nReturn JSON with keys answer, cite.\n"
            "Q: unused vacation payout for DE contractors?\n"
            '### Response:\n{"answer":"I do not have a cited policy for that.","cite":null}'
        )
    },
    {
        "text": (
            "### Instruction:\nAcknowledge VIP outage in under 40 words.\n"
            "### Response:\nAcknowledged — VIP outage noted since 09:00 UTC; we are escalating and will update within 30 minutes."
        )
    },
] * 24
ds = Dataset.from_list(rows)
args = TrainingArguments(
    output_dir="/content/peft_out",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=5,
    fp16=True,
    report_to=[],
)
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    tokenizer=tok,
    args=args,
    dataset_text_field="text",
    max_seq_length=256,
)
train_out = trainer.train()
adapter_dir = Path("/content/peft_out/adapter")
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)
print("train_loss", train_out.training_loss)
print("S3 (adapter) eval...")
s3 = eval_schema(model)
print("S3_schema_pass", s3)

In [ ]:
peft_receipt = {
    "status": "gpu",
    "cuda": True,
    "honesty": "CUDA PEFT micro-run on Colab with measured schema-pass delta — not peft_smoke. Not a DomainForge 7B S0/S3/S4 hire-depth ladder; use RunPod DomainForge pipeline for that.",
    "run_id": f"peft-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}",
    "base_model": BASE,
    "gpu": GPU_NAME,
    "cuda_device": GPU_NAME,
    "sft_examples": len(ds),
    "dpo_pairs": 0,
    "metrics": {
        "S0_schema_pass": round(s0, 4),
        "S3_schema_pass": round(s3, 4),
        "S4_schema_pass": round(s3, 4),
        "S4_preference_win_rate": 0.0,
        "train_loss": float(train_out.training_loss),
    },
    "adapter_uri": str(adapter_dir),
    "nvidia_smi_excerpt": SMI[:4000],
    "notes": "Colab Path A micro-receipt. S4 mirrors S3 (no DPO in this notebook).",
}
peft_path = OUT / "peft_gpu.json"
peft_path.write_text(json.dumps(peft_receipt, indent=2) + "\n")
print("wrote", peft_path)
print(json.dumps(peft_receipt, indent=2)[:900])

## Publish PEFT receipt

1. Download `peft_gpu.json`
2. From `modelforge-llmops` root:
   ```bash
   bash scripts/ingest_peft_gpu_receipt.sh ~/Downloads/peft_gpu.json
   git add docs/receipts/peft_gpu.json ui/public/receipts/peft_gpu.json
   git commit -m "Publish CUDA PEFT micro-receipt from Colab."
   git push origin main && cd ui && vercel --prod --yes
   ```
3. For **`vllm_cuda.json`**, still run RunPod: `bash scripts/one_shot_gpu_receipts.sh`
4. Confirm `/api/v1/posture` — PEFT should flip to ready; vLLM stays planned until upstream CUDA metrics


In [ ]:
from google.colab import files
files.download(str(peft_path))